# REGLab legal QA — deep dive for RAG teams

This notebook is written for people who need **more than histograms**: crosstabs, statistical tests, ECDFs, joint densities, and explicit **“so what?”** call-outs for retrieval/LLM budget design. Both datasets are **CC BY-SA 4.0** (see Hugging Face cards).

| Dataset | Card | Core RAG story |
|--------|------|----------------|
| Bar Exam (MBE-style MCQ + ~900k passages) | [reglab/barexam_qa](https://huggingface.co/datasets/reglab/barexam_qa) | Huge haystack; gold passage length ≠ random corpus snippet |
| Housing (state yes/no + ~1.7M statutes, 2021 law) | [reglab/housing_qa](https://huggingface.co/datasets/reglab/housing_qa) | **State** is a confounder; `questions_aux` drops corpus IDs needed for strict RAG |

> **فارسی:** هدف این نوت‌بوک «آشنا کردن جدی» با داده است: جداول (درصد گم‌شدگی، آزمون تعادل، چندک‌ها، همپوشانی `gold_idx` / `statute_idx`) و نمودارهایی که **پیام طراحی** بدهند—نه فقط چند نمودار سطحی.

**Why not `datasets`?** On Alliance / Compute Canada, PyArrow is often a dummy wheel; we load **CSV / TSV / zipped JSON** over HTTPS with pandas (same URLs the HF loader uses).

**Before running:** `module load scipy-stack/2024b`.

- **Interactive:** Jupyter on Narval often lacks `jupyter-lab`; use **VS Code / Cursor** remote with the same module loaded, or install Lab in a venv (heavy).
- **Headless (recommended on login/compute):** use the project venv + `nbconvert` — see `scripts/run_eda_notebook.sh` or:
  `bash scripts/run_eda_notebook.sh` → writes `notebooks/eda_reglab_deep_dive_executed.ipynb` with all figure outputs embedded.

### What to read in order
1. **Part A — Bar Exam:** scale table → missingness → χ² on answer balance → **prompt sharing** → ECDF lengths → **gold vs corpus** quantiles → **joint density & Spearman** → passage `gold_idx` hit-rate → source×subject heatmap → **Cramér V** (source vs answer).
2. **Part B — Housing:** eval vs aux → state Yes-rates (with filters) → cross-state heterogeneity by `question_group` → excerpt counts vs label → **gold excerpts vs full statute ECDF** → **`statute_idx` vs corpus-sample overlap**.
3. **Takeaways:** checklist for indexing, conditioning, and evaluation splits.


In [1]:
from __future__ import annotations

import io
import json
import math
import urllib.request
import zipfile
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless/fresh execute (nbconvert, compute nodes)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import seaborn as sns

    sns.set_theme(style="whitegrid", context="talk", font_scale=0.85)
    HAS_SNS = True
except ImportError:
    HAS_SNS = False
    plt.style.use("seaborn-v0_8-whitegrid")

from scipy import stats

# --- URLs (pinned to `main` on HF) ---
BAREXAM_QA_CSV = (
    "https://huggingface.co/datasets/reglab/barexam_qa/resolve/main/data/qa/qa.csv"
)
BAREXAM_PASSAGES_TSV = (
    "https://huggingface.co/datasets/reglab/barexam_qa/resolve/main/data/passages/passages.tsv"
)
HOUSING_Q = "https://huggingface.co/datasets/reglab/housing_qa/resolve/main/data/questions.json.zip"
HOUSING_Q_AUX = "https://huggingface.co/datasets/reglab/housing_qa/resolve/main/data/questions_aux.json.zip"
HOUSING_STATUTES = (
    "https://huggingface.co/datasets/reglab/housing_qa/resolve/main/data/statutes.tsv"
)


def _len_str(x) -> int:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return 0
    return len(str(x))


def fetch_json_zip(url: str) -> list[dict]:
    raw = urllib.request.urlopen(url, timeout=300).read()
    zf = zipfile.ZipFile(io.BytesIO(raw))
    name = zf.namelist()[0]
    return json.loads(zf.read(name).decode("utf-8"))


def ecdf(a: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return sorted x and F(x) for a 1-D array (handles duplicates)."""
    x = np.sort(a.astype(float))
    n = len(x)
    if n == 0:
        return x, x
    y = np.arange(1, n + 1) / n
    return x, y


def show_df(df: pd.DataFrame, title: str | None = None) -> None:
    if title:
        display(Markdown(f"**{title}**"))
    try:
        styler = (
            df.style.set_properties(**{"text-align": "right"})
            .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
            .format(na_rep="—")
        )
        display(styler)
    except Exception:
        display(df)


PASSAGE_SAMPLE_ROWS = 200_000   # raise on a fat node if you want less sampling error
STATUTE_SAMPLE_ROWS = 250_000
print("Passage/statute samples:", PASSAGE_SAMPLE_ROWS, STATUTE_SAMPLE_ROWS)


Passage/statute samples: 200000 250000


## Part A — Bar Exam QA (`qa` + passage pool)

**Task shape:** multiple-choice with **gold passage** text / `gold_idx`. The **passage corpus is ~900k** snippets; retrieval must find one needle in a very large haystack.

**Columns worth internalizing:** `prompt` (shared stem), `question`, four choices, `answer`, `gold_passage`, metadata (`source`, `subject`, `prompt_id`).


In [2]:
print("Loading Bar Exam qa.csv …")
bar_qa = pd.read_csv(BAREXAM_QA_CSV)
bar_qa["answer"] = bar_qa["answer"].astype(str).str.upper().str.strip()

for c in ["prompt", "question", "choice_a", "choice_b", "choice_c", "choice_d", "gold_passage"]:
    bar_qa[f"{c}_nchars"] = bar_qa[c].map(_len_str)

bar_qa["choices_total_nchars"] = (
    bar_qa["choice_a_nchars"]
    + bar_qa["choice_b_nchars"]
    + bar_qa["choice_c_nchars"]
    + bar_qa["choice_d_nchars"]
)

overview_rows = {
    "rows": len(bar_qa),
    "unique_idx": bar_qa["idx"].nunique(),
    "unique_prompt_id": bar_qa["prompt_id"].nunique(),
    "unique_gold_idx": bar_qa["gold_idx"].nunique(),
    "sources": bar_qa["source"].nunique(),
    "subjects_non_missing": bar_qa["subject"].notna().sum(),
}
overview = pd.Series(overview_rows, name="value").to_frame()
show_df(overview, "Dataset scale (QA split / bundled CSV)")

miss = (bar_qa.isna().mean().sort_values(ascending=False) * 100).rename("missing_%").to_frame()
show_df(miss.head(12), "Top columns by missing % (bar exam QA)")

pct = bar_qa[[
    "prompt_nchars", "question_nchars", "choices_total_nchars", "gold_passage_nchars"
]].agg(["mean", "std", "median", "min", lambda s: s.quantile(0.9), "max"])
pct.index = ["mean", "std", "median", "min", "p90", "max"]
show_df(pct, "Text length (characters) — key fields")


Loading Bar Exam qa.csv …


**Dataset scale (QA split / bundled CSV)**

,value
rows,1195
unique_idx,1195
unique_prompt_id,935
unique_gold_idx,1149
sources,13
subjects_non_missing,594


**Top columns by missing % (bar exam QA)**

,missing_%
prompt,62.761506
subject,50.292887
choice_d,0.083682
idx,0.000000
prompt_id,0.000000
example_id,0.000000
source,0.000000
dataset,0.000000
question_number,0.000000
question,0.000000


**Text length (characters) — key fields**

,prompt_nchars,question_nchars,choices_total_nchars,gold_passage_nchars
mean,298.669456,532.289540,358.272803,742.541423
std,473.199600,403.444783,183.037957,729.434829
median,0.000000,443.000000,332.000000,569.000000
min,0.000000,20.000000,24.000000,42.000000
p90,1112.000000,1127.600000,581.600000,1342.200000
max,1870.000000,2144.000000,1328.000000,10998.000000


In [3]:
# Answer distribution vs uniform (MCQs often intentionally balanced)
counts = bar_qa["answer"].value_counts().reindex(["A", "B", "C", "D"]).fillna(0).astype(int)
expected = np.full(4, len(bar_qa) / 4)
chi2, p = stats.chisquare(counts.values, expected)
tab = pd.DataFrame({"count": counts, "uniform_expected": expected}).T
show_df(tab, f"Answer counts · χ² vs uniform = {chi2:.2f}, p = {p:.3g} (high p ⇒ close to balanced)")

q_per_prompt = bar_qa.groupby("prompt_id", observed=True).size().rename("n_questions_same_prompt")
qpp_stats = q_per_prompt.agg(["count", "mean", "median", "max", lambda s: (s > 1).mean() * 100])
qpp_stats.index = ["n_distinct_prompt_ids", "mean_questions_per_prompt", "median_questions_per_prompt", "max_questions_per_prompt", "pct_prompts_with_gt1_question"]
show_df(qpp_stats.to_frame("value"), "Prompt sharing — bar exams reuse a fact pattern across several scored items")


**Answer counts · χ² vs uniform = 0.35, p = 0.95 (high p ⇒ close to balanced)**

answer,A,B,C,D
count,294.000000,295.000000,299.000000,307.000000
uniform_expected,298.750000,298.750000,298.750000,298.750000


**Prompt sharing — bar exams reuse a fact pattern across several scored items**

,value
n_distinct_prompt_ids,935.000000
mean_questions_per_prompt,1.278075
median_questions_per_prompt,1.000000
max_questions_per_prompt,6.000000
pct_prompts_with_gt1_question,19.572193


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

axes[0, 0].bar(counts.index, counts.values, color=["#2b6cb0", "#2f855a", "#c05621", "#805ad5"])
axes[0, 0].axhline(len(bar_qa) / 4, color="black", ls="--", lw=1, label="perfect balance")
axes[0, 0].set_title("Correct label distribution")
axes[0, 0].legend()
axes[0, 0].set_xlabel("answer key")

qpp = q_per_prompt.values
axes[0, 1].hist(qpp, bins=np.arange(0.5, qpp.max() + 1.5), color="#2c5282", edgecolor="white")
axes[0, 1].set_title("Questions per `prompt_id`\n(shared vignettes → multi-hop / shared-context modeling)")
axes[0, 1].set_xlabel("# questions sharing the same prompt")
axes[0, 1].set_ylabel("count")

src = bar_qa["source"].astype(str).value_counts().head(14)
axes[1, 0].barh(src.index[::-1], src.values[::-1], color="#276749")
axes[1, 0].set_title("Top `source` values\n(exam form / booklet lineage)")

sub = bar_qa["subject"].fillna("(missing)").astype(str).value_counts().head(14)
axes[1, 1].barh(sub.index[::-1], sub.values[::-1], color="#b7791f")
axes[1, 1].set_title("Top `subject` (missing grouped)")

plt.suptitle("Bar Exam QA — structure & balance", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


/tmp/ipykernel_1889846/268058424.py:24: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


In [5]:
fig, ax = plt.subplots(figsize=(9, 5.5))
labels = {
    "prompt_nchars": "prompt (shared stem)",
    "question_nchars": "question (scored stem)",
    "choices_total_nchars": "all four choices (sum)",
    "gold_passage_nchars": "gold passage (supervision)",
}
palette = ["#3182ce", "#38a169", "#dd6b20", "#805ad5"]
for (col, lab), color in zip(labels.items(), palette):
    x, y = ecdf(bar_qa[col].values)
    ax.plot(x, y, lw=2.2, label=lab, color=color)

ax.set_title("ECDF of text lengths — what token budgets should RAG / LLM expect?")
ax.set_xlabel("characters")
ax.set_ylabel("F(x)")
ax.legend(loc="lower right", fontsize=10)
plt.tight_layout()
plt.show()

# Short insight table: share of gold passages longer than corpus median (after we load passages)
print("Gold passage len — quartiles:", bar_qa["gold_passage_nchars"].quantile([0.25, 0.5, 0.75]).to_dict())


Gold passage len — quartiles: {0.25: 362.0, 0.5: 569.0, 0.75: 910.0}


### Joint behavior of text lengths (bar exam)

**Why this section:** Scatter / hexbin and Spearman correlation show whether “long questions” come with “long gold passages” (chunk/rerank budget) and whether all four choices jointly grow with the stem—insights you do not get from four separate histograms.

In [6]:
cols_len = [
    "prompt_nchars",
    "question_nchars",
    "choices_total_nchars",
    "gold_passage_nchars",
]
rho = bar_qa[cols_len].corr(method="spearman")
show_df(rho, "Spearman ρ — text lengths (robust to heavy tails; bar exam QA)")

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
ax = axes[0]
if HAS_SNS:
    sns.heatmap(
        rho,
        ax=ax,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0.0,
        vmin=-1,
        vmax=1,
        square=True,
    )
else:
    im = ax.imshow(rho.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(rho.columns)))
    ax.set_yticks(range(len(rho.index)))
    ax.set_xticklabels(rho.columns, rotation=35, ha="right")
    ax.set_yticklabels(rho.index)
    plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Correlation heatmap\n(long stems vs long supervision?)")

ax2 = axes[1]
x = bar_qa["question_nchars"].to_numpy()
y = bar_qa["gold_passage_nchars"].to_numpy()
hb = ax2.hexbin(x, y, gridsize=38, cmap="magma", mincnt=1, linewidths=0)
plt.colorbar(hb, ax=ax2, label="count per hex bin")
ax2.set_xlabel("question length (chars)")
ax2.set_ylabel("gold passage length (chars)")
ax2.set_title("Joint mass: scored stem vs gold passage\n(fat upper tail ⇒ plan evidence windows)")

plt.tight_layout()
plt.show()

display(
    Markdown(
        "**Interpretation cue:** weak off-diagonal structure means you cannot infer gold span size from "
        "stem size alone—retrieval should not assume a single fixed chunk length matches annotated evidence."
    )
)

**Spearman ρ — text lengths (robust to heavy tails; bar exam QA)**

,prompt_nchars,question_nchars,choices_total_nchars,gold_passage_nchars
prompt_nchars,1.000000,-0.688227,-0.045394,-0.020258
question_nchars,-0.688227,1.000000,0.124663,0.088824
choices_total_nchars,-0.045394,0.124663,1.000000,0.088957
gold_passage_nchars,-0.020258,0.088824,0.088957,1.000000


/tmp/ipykernel_1889846/1779396415.py:42: UserWarning: Glyph 8658 (\N{RIGHTWARDS DOUBLE ARROW}) missing from font(s) Liberation Sans.
  plt.tight_layout()


**Interpretation cue:** weak off-diagonal structure means you cannot infer gold span size from stem size alone—retrieval should not assume a single fixed chunk length matches annotated evidence.

In [7]:
print(f"Reading passages.tsv (first {PASSAGE_SAMPLE_ROWS:,} rows) …")
psg = pd.read_csv(BAREXAM_PASSAGES_TSV, sep="\t", nrows=PASSAGE_SAMPLE_ROWS)
psg["text_nchars"] = psg["text"].map(_len_str)

g_ecdf_x, g_ecdf_y = ecdf(bar_qa["gold_passage_nchars"].values)
c_ecdf_x, c_ecdf_y = ecdf(psg["text_nchars"].values)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(g_ecdf_x, g_ecdf_y, lw=2.5, label="gold passages (training labels)", color="#c53030")
ax.plot(c_ecdf_x, c_ecdf_y, lw=2.5, label=f"passage pool sample (n={len(psg):,})", color="#2b6cb0")
ax.set_xscale("log")
ax.set_title(
    "ECDF comparison: gold passages vs. random corpus prefix\n"
    "(log-x emphasizes heavy-tailed length; compare medians visually)"
)
ax.set_xlabel("characters (log scale)")
ax.set_ylabel("F(x)")
ax.legend()
plt.tight_layout()
plt.show()

qs = np.linspace(0, 1, 9)
comp = pd.DataFrame({
    "quantile": qs,
    "gold_passage": np.quantile(bar_qa["gold_passage_nchars"], qs),
    "corpus_sample": np.quantile(psg["text_nchars"], qs),
})
show_df(comp, "Parallel quantiles — gold vs passage-pool sample (same rows order as quantile grid)")

src_mix = psg["source"].astype(str).value_counts(normalize=True).mul(100).round(2)
show_df(src_mix.to_frame("pct_of_sample"), "Passage-pool sample: source mix (sequential read — stable mix at large n)")


Reading passages.tsv (first 200,000 rows) …


**Parallel quantiles — gold vs passage-pool sample (same rows order as quantile grid)**

,quantile,gold_passage,corpus_sample
0,0.000000,42.000000,17.000000
1,0.125000,263.750000,136.000000
2,0.250000,362.000000,245.000000
3,0.375000,445.500000,359.000000
4,0.500000,569.000000,476.000000
5,0.625000,721.000000,607.000000
6,0.750000,910.000000,772.000000
7,0.875000,1222.000000,1024.000000
8,1.000000,10998.000000,276898.000000


**Passage-pool sample: source mix (sequential read — stable mix at large n)**

,pct_of_sample
source,
caselaw,95.200000
wex,3.360000
mbe,1.450000


In [8]:
# Retrieval-relevant join sanity: is `gold_idx` in the pool prefix?
hit = bar_qa["gold_idx"].isin(psg["idx"].astype(str))
hit_rate = hit.mean() * 100
display(Markdown(
    f"**Gold idx overlap with corpus prefix:** {hit.sum()} / {len(bar_qa)} "
    f"({hit_rate:.1f}%) of QA rows have `gold_idx` present in the first {len(psg):,} passage rows.\n\n"
    "If this is <100%, increase `PASSAGE_SAMPLE_ROWS` or stream the full TSV—otherwise your sampled ECDF mixes two populations."
))

# Optional: source of matched gold rows (if any)
if hit.any():
    merged = bar_qa.loc[hit].merge(
        psg.drop_duplicates("idx")[["idx", "source"]],
        left_on="gold_idx", right_on="idx", how="left", suffixes=("", "_pool"),
    )
    show_df(
        merged["source_pool"].astype(str).value_counts().head(10).to_frame("n"),
        "Sources for gold passages found in pool prefix",
    )


**Gold idx overlap with corpus prefix:** 1195 / 1195 (100.0%) of QA rows have `gold_idx` present in the first 200,000 passage rows.

If this is <100%, increase `PASSAGE_SAMPLE_ROWS` or stream the full TSV—otherwise your sampled ECDF mixes two populations.

**Sources for gold passages found in pool prefix**

,n
source_pool,
mbe,1195


## Part B — Housing QA

Three splits in the paper / HF card:

| Split | Role |
|-------|------|
| `questions` | **RAG-evaluable** yes/no with statute excerpts + `statute_idx` keys |
| `questions_aux` | More items, **no** full corpus linkage (weak supervision / LM-only) |
| `statutes` | Large statutory haystack (~1.7M rows scraped from Justia, 2021 snapshot) |

**Evaluation framing:** models should answer with respect to **2021** state law; coverage is uneven by state/question.


In [9]:
print("Loading Housing QA …")
hq = pd.DataFrame(fetch_json_zip(HOUSING_Q))
hq_aux = pd.DataFrame(fetch_json_zip(HOUSING_Q_AUX))
hq["answer_bin"] = (hq["answer"].astype(str).str.lower().str.strip() == "yes").astype(int)

hq["n_statute_excerpts"] = hq["statutes"].map(lambda s: len(s) if isinstance(s, list) else 0)
hq["excerpt_chars_total"] = hq["statutes"].map(
    lambda sts: sum(_len_str(x.get("excerpt")) for x in sts) if isinstance(sts, list) else 0
)

summ = pd.Series(
    {
        "questions_rows": len(hq),
        "questions_aux_rows": len(hq_aux),
        "n_states_in_questions": hq["state"].nunique(),
        "n_question_groups": hq["question_group"].nunique(),
        "mean_statute_excerpts": hq["n_statute_excerpts"].mean(),
        "median_excerpt_chars_total": hq["excerpt_chars_total"].median(),
        "overall_yes_rate_%": hq["answer_bin"].mean() * 100,
    },
    name="value",
).to_frame()
show_df(summ, "Housing QA — high-level")

aux_ids = set(hq_aux["idx"].astype(int))
q_ids = set(hq["idx"].astype(int))
display(Markdown(
    f"**Aux vs eval overlap:** `questions` idx ⊆ `questions_aux` check — "
    f"{len(q_ids - aux_ids)} ids only in questions; {len(aux_ids - q_ids):,} extra rows in aux only."
))

imb = (
    hq.groupby("state", observed=True)
    .agg(n=("answer_bin", "size"), yes_rate=("answer_bin", "mean"))
    .assign(yes_rate=lambda d: (d["yes_rate"] * 100).round(1))
    .sort_values("n", ascending=False)
)
show_df(imb.head(15), "Largest states by n (showing marginal Yes %)")
show_df(imb.sort_values("yes_rate", ascending=False).head(10), "Highest Yes-rate states (watch small n)")
show_df(imb.sort_values("yes_rate").head(10), "Lowest Yes-rate states (watch small n)")


Loading Housing QA …


**Housing QA — high-level**

,value
questions_rows,6853.000000
questions_aux_rows,9297.000000
n_states_in_questions,48.000000
n_question_groups,226.000000
mean_statute_excerpts,2.606450
median_excerpt_chars_total,1323.000000
overall_yes_rate_%,35.225449


**Aux vs eval overlap:** `questions` idx ⊆ `questions_aux` check — 0 ids only in questions; 2,444 extra rows in aux only.

**Largest states by n (showing marginal Yes %)**

,n,yes_rate
state,,
Washington,201,38.300000
Colorado,192,35.400000
New York,190,38.900000
Connecticut,182,43.400000
California,181,42.500000
Wisconsin,176,37.500000
North Carolina,175,28.000000
Delaware,174,33.900000
Virginia,174,35.100000


**Highest Yes-rate states (watch small n)**

,n,yes_rate
state,,
Alaska,92,48.900000
District of Columbia,119,45.400000
Arizona,69,43.500000
Connecticut,182,43.400000
California,181,42.500000
Rhode Island,170,40.600000
Oregon,154,40.300000
Montana,127,40.200000
New Hampshire,174,39.100000


**Lowest Yes-rate states (watch small n)**

,n,yes_rate
state,,
Puerto Rico,108,24.100000
Wyoming,99,26.300000
Missouri,166,27.100000
Illinois,148,27.700000
North Carolina,175,28.000000
South Dakota,151,28.500000
Maryland,161,28.600000
Idaho,114,29.800000
Indiana,156,30.100000


In [10]:
# Within the same question_group, answers can disagree across states → jurisprudence signal
rows = []
for g, sub in hq.groupby("question_group", observed=True):
    rates = sub.groupby("state")["answer_bin"].mean()
    rows.append(
        {
            "question_group": g,
            "n_states": rates.size,
            "mean_yes_rate": rates.mean(),
            "std_yes_rate": rates.std(ddof=0),
        }
    )
qg_spread = pd.DataFrame(rows).dropna(subset=["std_yes_rate"])
qg_spread = qg_spread.sort_values("std_yes_rate", ascending=False)
show_df(qg_spread.head(12), "Most *heterogeneous* question groups across states (high std of Yes-rate)")

fig, ax = plt.subplots(figsize=(9, 5.5))
sub = qg_spread.head(18).sort_values("std_yes_rate")
ax.barh(sub["question_group"].astype(str), sub["std_yes_rate"], color="#654c8f")
ax.set_xlabel("std of state-level Yes rate (within question_group)")
ax.set_title("Where housing law diverges across jurisdictions (same LSC question template)")
plt.tight_layout()
plt.show()


**Most *heterogeneous* question groups across states (high std of Yes-rate)**

,question_group,n_states,mean_yes_rate,std_yes_rate
92,92,40,0.500000,0.500000
49,49,42,0.500000,0.500000
193,193,42,0.500000,0.500000
125,125,42,0.500000,0.500000
112,112,2,0.500000,0.500000
98,98,2,0.500000,0.500000
147,147,43,0.511628,0.499865
181,181,42,0.476190,0.499433
157,157,40,0.475000,0.499375
46,46,36,0.527778,0.499228


In [11]:
# Reliable marginal Yes rates: filter states with enough n
MIN_N = 60
marg = (
    hq.groupby("state", observed=True)
    .agg(n=("answer_bin", "size"), yes_rate=("answer_bin", "mean"))
    .query("n >= @MIN_N")
    .assign(yes_rate=lambda d: d["yes_rate"] * 100)
    .sort_values("yes_rate")
)

fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(marg))))
ax.barh(marg.index, marg["yes_rate"], color=np.where(marg["yes_rate"] > 50, "#2f855a", "#c53030"))
ax.axvline(50, color="black", lw=1, ls=":")
ax.set_xlabel(f'Yes rate (%) — states with n ≥ {MIN_N}')
ax.set_title("Housing QA — jurisdiction skew (2021 law framing)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
parts = ax.violinplot(
    [hq.loc[hq["answer_bin"] == 0, "n_statute_excerpts"].dropna(),
     hq.loc[hq["answer_bin"] == 1, "n_statute_excerpts"].dropna()],
    positions=[0, 1],
    showmeans=True,
    showmedians=True,
)
ax.set_xticks([0, 1])
ax.set_xticklabels(["No", "Yes"])
ax.set_ylabel("# annotated statute excerpts in label")
ax.set_title("Annotation load vs label (not causal — designers may cite more for harder Yes cases)")
plt.tight_layout()
plt.show()


In [12]:
print(f"Statutes.tsv sample (first {STATUTE_SAMPLE_ROWS:,}) …")
st = pd.read_csv(HOUSING_STATUTES, sep="\t", nrows=STATUTE_SAMPLE_ROWS)
st["text_nchars"] = st["text"].map(_len_str)

ex_lens = []
for sts in hq["statutes"]:
    if not isinstance(sts, list):
        continue
    for x in sts:
        ex_lens.append(_len_str(x.get("excerpt")))
ex_lens = np.asarray(ex_lens, dtype=int)

fig, ax = plt.subplots(figsize=(9, 5.5))
gx, gy = ecdf(ex_lens)
sx, sy = ecdf(st["text_nchars"].values)
ax.plot(gx, gy, lw=2.4, label="gold excerpts attached to questions", color="#c53030")
ax.plot(sx, sy, lw=2.4, label=f"statute corpus sample (n={len(st):,})", color="#2b6cb0")
ax.set_xscale("log")
ax.set_title("ECDF: annotated excerpts vs statute corpus (log chars)\n"
             "Corpus entries are full statute blocks; snippets in questions are shorter, targeted pulls.")
ax.set_xlabel("characters")
ax.set_ylabel("F(x)")
ax.legend()
plt.tight_layout()
plt.show()

st_pct = pd.DataFrame({
    "field": ["gold_excerpts_in_questions", "statute_corpus_sample"],
    "p50": [np.median(ex_lens), st["text_nchars"].median()],
    "p90": [np.quantile(ex_lens, 0.9), st["text_nchars"].quantile(0.9)],
    "p99": [np.quantile(ex_lens, 0.99), st["text_nchars"].quantile(0.99)],
})
show_df(st_pct.set_index("field"), "Length landmarks — design chunking & context windows")

display(Markdown(
    "**Note:** statute TSV is read sequentially; the state histogram on the sample is **not** a uniform draw "
    "from the full 1.7M rows—use it for lengths / schema, not exact prevalence by state."
))


Statutes.tsv sample (first 250,000) …


**Length landmarks — design chunking & context windows**

,p50,p90,p99
field,,,
gold_excerpts_in_questions,556.000000,2025.000000,5779.000000
statute_corpus_sample,881.000000,3824.000000,13969.070000


**Note:** statute TSV is read sequentially; the state histogram on the sample is **not** a uniform draw from the full 1.7M rows—use it for lengths / schema, not exact prevalence by state.

### Housing — `statute_idx` vs your corpus sample

**Message:** RAG evaluation joins **question-level** gold keys to rows in `statutes`. If you only loaded the **first N** lines of `statutes.tsv`, reported recall@k can be optimistic or pessimistic depending on whether those keys landed in the prefix—this table quantifies that coverage for your chosen `STATUTE_SAMPLE_ROWS`.

In [13]:
idx_in_labels: set[int] = set()
rows_with_key = 0
for sts in hq["statutes"]:
    if not isinstance(sts, list):
        continue
    row_had = False
    for x in sts:
        if not isinstance(x, dict) or x.get("statute_idx") is None:
            continue
        try:
            idx_in_labels.add(int(x["statute_idx"]))
            row_had = True
        except (TypeError, ValueError):
            continue
    if row_had:
        rows_with_key += 1

pool_ids = set(st["idx"].astype(int))
inter = idx_in_labels & pool_ids

summ = pd.Series(
    {
        "questions_rows": len(hq),
        "questions_with_any_statute_idx": rows_with_key,
        "unique_statute_idx_in_labels": len(idx_in_labels),
        "corpus_sample_unique_idx": int(st["idx"].nunique()),
        "label_ids_also_in_corpus_sample": len(inter),
        "pct_label_ids_hit_in_sample": round(100.0 * len(inter) / max(len(idx_in_labels), 1), 2),
    },
    name="value",
).to_frame()
show_df(summ, "Coverage of gold statute IDs inside the loaded `statutes` prefix")

display(
    Markdown(
        "**Action:** if `pct_label_ids_hit_in_sample` is not ~100, bump `STATUTE_SAMPLE_ROWS` for notebook fidelity "
        "or stream the full TSV once when building the retrieval index."
    )
)

**Coverage of gold statute IDs inside the loaded `statutes` prefix**

,value
questions_rows,6853.000000
questions_with_any_statute_idx,6853.000000
unique_statute_idx_in_labels,990.000000
corpus_sample_unique_idx,250000.000000
label_ids_also_in_corpus_sample,132.000000
pct_label_ids_hit_in_sample,13.330000


**Action:** if `pct_label_ids_hit_in_sample` is not ~100, bump `STATUTE_SAMPLE_ROWS` for notebook fidelity or stream the full TSV once when building the retrieval index.

### Extra: where do items concentrate? (`source` × `subject`)

Heatmaps compress big crosstabs—useful when pitching *coverage* to collaborators.


In [14]:
# Crosstab of exam form × subject (top slices only for readability)
subj = bar_qa["subject"].fillna("(missing)").astype(str)
TOP_SRC = 10
TOP_SUBJ = 10
src_top = bar_qa["source"].astype(str).value_counts().head(TOP_SRC).index
sub_top = subj.value_counts().head(TOP_SUBJ).index
mask = bar_qa["source"].astype(str).isin(src_top) & subj.isin(sub_top)
ct = pd.crosstab(bar_qa.loc[mask, "source"].astype(str), subj.loc[mask])

fig, ax = plt.subplots(figsize=(11, 5.5))
mat = ct.values.astype(float)
im = ax.imshow(mat, aspect="auto", cmap="Blues", interpolation="nearest")
ax.set_xticks(np.arange(ct.shape[1]))
ax.set_xticklabels(ct.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(ct.shape[0]))
ax.set_yticklabels(ct.index)
vmax = mat.max() if mat.size else 1.0
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        v = int(mat[i, j])
        if v == 0:
            continue
        ax.text(
            j,
            i,
            str(v),
            ha="center",
            va="center",
            color="white" if v > vmax * 0.55 else "#1a202c",
            fontsize=8,
        )
plt.colorbar(im, ax=ax, label="count")
ax.set_title("Bar Exam QA — counts for top sources × top subjects")
plt.tight_layout()
plt.show()

show_df(ct, "Raw counts table (same as heatmap)")


**Raw counts table (same as heatmap)**

subject,(missing),CONST. LAW,CONTRACTS,CRIM. LAW,EVIDENCE,REAL PROP.,TORTS
source,,,,,,,
1991-Feb,0,30,39,29,30,29,40
1991-July,0,30,40,28,30,30,39
1998,0,33,34,32,33,33,33
MBE-1972-78-part2,50,0,0,0,0,0,0
MBE-1972-78-part3,51,0,0,0,0,0,0
MBE-1972-78-part4,51,0,0,0,0,0,0
MBE-1978-83-part1,197,0,0,0,0,0,0
MBE-1978-83-part5,52,0,0,0,0,0,0
MBE-1978-83-part6,52,0,0,0,0,0,0


In [15]:
def cramers_v(table: pd.DataFrame) -> float:
    """Association strength for categorical×categorical tables in [0, 1]."""
    chi2, _, _, _ = stats.chi2_contingency(table, correction=False)
    n = table.values.sum()
    r, k = table.shape
    if n == 0 or min(r, k) < 2:
        return float("nan")
    return math.sqrt(chi2 / (n * (min(k, r) - 1)))


top_src = bar_qa["source"].astype(str).value_counts().head(6).index
sub = bar_qa.loc[bar_qa["source"].astype(str).isin(top_src), ["source", "answer"]]
ct = pd.crosstab(sub["source"].astype(str), sub["answer"].astype(str))
v = cramers_v(ct)
show_df(ct, f"Top-6 `source` × answer · Cramér V ≈ {v:.3f} (0 ⇒ no association)")

fig, ax = plt.subplots(figsize=(9.5, 4.2))
row_pct = ct.div(ct.sum(axis=1), axis=0) * 100.0
row_pct.plot(kind="barh", stacked=True, ax=ax, colormap="viridis", width=0.86, legend=True)
ax.set_xlabel("within-source % of items")
ax.set_title("Answer mix conditional on booklet/source\n(useful to catch exam-form leakage / imbalance)")
ax.legend(title="answer", bbox_to_anchor=(1.02, 1), frameon=False)
plt.tight_layout()
plt.show()

**Top-6 `source` × answer · Cramér V ≈ 0.021 (0 ⇒ no association)**

answer,A,B,C,D
source,,,,
1991-Feb,51,48,50,48
1991-July,50,48,49,50
1998,48,50,49,51
MBE-1978-83-part1,48,48,53,48
MBE-1978-83-part5,12,14,12,14
MBE-1978-83-part6,12,12,14,14


## Takeaways for building Legal RAG

1. **Bar exam:** Labels are usually **near-balanced** across A–D; many questions **share a prompt** (`prompt_id`)—plan batching / long-context vs retrieve-then-read accordingly. **Gold passages** sit on a different length distribution than generic corpus snippets—chunking tuned on the corpus alone can hurt the annotated evidence spans. **Spearman / hexbin** help stress-test the “fixed chunk size” assumption; **Cramér V** checks whether answer leakage tracks exam *booklet* (`source`) rather than law difficulty.
2. **Housing:** Treat **state + year (2021)** as first-class conditioning; the same `question_group` can flip across jurisdictions—naive monolithic indexes blending all states will confound retrieval. **`questions_aux`** scales LM-only experiments but **drops corpus IDs** needed for strict RAG evaluation on the big `statutes` split. Use the **`statute_idx` coverage table** before trusting any retrieval metric computed on a prefix of `statutes-only.tsv`.
3. **Deployment caution:** both cards stress licensing & non-advice use—mirror that in READMEs if you ship demos.
